# Prediction des matchs - Coupe du Monde 2026

Notebook de base pour recuperer des donnees football, preparer les features, puis entrainer un modele de prediction.

## 1) Setup et imports
Assure-toi d'avoir une variable `API_FOOTBALL_KEY` dans ton fichier `.env`.

In [2]:
import os
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_FOOTBALL_KEY")

if API_KEY:
    print("API key detectee.")
else:
    print("API key manquante: ajoute API_FOOTBALL_KEY dans .env")

API key detectee.


## 2) Donnees du jour depuis API-FOOTBALL
Cette cellule appelle l API pour recuperer les matchs de la date du jour (UTC).

In [4]:
from datetime import datetime, timezone

def fetch_match_data(date: str, api_key: str) -> pd.DataFrame:
    if not api_key:
        raise ValueError("API_FOOTBALL_KEY est manquante. Ajoute-la dans .env.")

    url = "https://v3.football.api-sports.io/fixtures"
    headers = {
        "x-apisports-key": api_key,
    }
    params = {"date": date}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()

    fixtures = payload.get("response", [])
    rows = []
    for item in fixtures:
        fixture = item.get("fixture", {})
        league = item.get("league", {})
        teams = item.get("teams", {})
        goals = item.get("goals", {})
        score = item.get("score", {})

        home_goals = goals.get("home")
        away_goals = goals.get("away")

        rows.append(
            {
                "Match_ID": fixture.get("id"),
                "Date": fixture.get("date"),
                "Status": (fixture.get("status") or {}).get("short"),
                "League": league.get("name"),
                "Home": (teams.get("home") or {}).get("name"),
                "Away": (teams.get("away") or {}).get("name"),
                "HomeGoals": home_goals if home_goals is not None else 0,
                "AwayGoals": away_goals if away_goals is not None else 0,
                "HomeGoalsRaw": home_goals,
                "AwayGoalsRaw": away_goals,
                "HT_Home": ((score.get("halftime") or {}).get("home")),
                "HT_Away": ((score.get("halftime") or {}).get("away")),
            }
        )

    return pd.DataFrame(rows)

today_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d")
df_all = fetch_match_data(today_utc, API_KEY)

# Garde uniquement les matchs de la competition "World Cup"
df = df_all[df_all["League"].fillna("").str.casefold().eq("world cup")].reset_index(drop=True)

print(f"Matchs World Cup recuperes pour {today_utc}: {len(df)} / {len(df_all)}")
df.head(20)

Matchs World Cup recuperes pour 2026-06-13: 3 / 570


,Match_ID,Date,Status,League,Home,Away,HomeGoals,AwayGoals,HomeGoalsRaw,AwayGoalsRaw,HT_Home,HT_Away
0,1489370,2026-06-13T01:00:00+00:00,FT,World Cup,USA,Paraguay,4,1,4.0,1.0,3.0,0.0
1,1489373,2026-06-13T19:00:00+00:00,NS,World Cup,Qatar,Switzerland,0,0,NaN,NaN,NaN,NaN
2,1489371,2026-06-13T22:00:00+00:00,NS,World Cup,Brazil,Morocco,0,0,NaN,NaN,NaN,NaN


## 3) Feature engineering simple

In [ ]:
df = df.copy()
df["GoalDiff"] = df["HomeGoals"] - df["AwayGoals"]
df["HomeWin"] = (df["GoalDiff"] > 0).astype(int)
df

## 4) Baseline model (exemple)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df[["HomeGoals", "AwayGoals", "GoalDiff"]]
y = df["HomeWin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.34, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))